# Task 2: Original LCS System on Raw Dataset

## Imports

In [87]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import chi2_contingency
from skeLCS import eLCS
import time
import re



## Loading Dataset: Using Pandas

Loading the raw dataset without any preprocessing. This is not yet suitable for an LCS baseline. 

In [32]:
# Read from CSV file
data = pd.read_csv("/Users/christian/Uni_Files/2026/Semester 2/Data Engineering and AI/Data-Engineering-and-AI-Project-Phase-1/Phase-I/PhiUSIIL_Phishing_URL_Dataset.csv")

# Specify the dataset's phenotype label
classLabel = "label"

# Derive the attributes and phenotype array using the phenotype label
dataFeatures = data.drop(classLabel, axis= 1).values
dataPhenotypes = data[classLabel].values

dataHeaders = data.drop(classLabel, axis=1).columns.values

print("Data Features")
print(dataFeatures)
print("Data Phenotypes")
print(dataPhenotypes)
print("Data Headers")
print(dataHeaders)

Data Features
[['521848.txt' 'https://www.southbankmosaics.com' 31 ... 119 0 124]
 ['31372.txt' 'https://www.uni-mainz.de' 23 ... 39 0 217]
 ['597387.txt' 'https://www.voicefmradio.co.uk' 29 ... 42 2 5]
 ...
 ['622132.txt' 'https://www.nononsensedesign.be' 30 ... 58 2 67]
 ['7503962.txt'
  'https://patient-cell-40f5.updatedlogmylogin.workers.dev/' 55 ... 0 0 0]
 ['384822.txt' 'https://www.alternativefinland.com' 33 ... 256 0 261]]
Data Phenotypes
[1 1 1 ... 1 0 1]
Data Headers
<StringArray>
[                  'FILENAME',                        'URL',
                  'URLLength',                     'Domain',
               'DomainLength',                 'IsDomainIP',
                        'TLD',         'URLSimilarityIndex',
       'CharContinuationRate',          'TLDLegitimateProb',
                'URLCharProb',                  'TLDLength',
              'NoOfSubDomain',             'HasObfuscation',
         'NoOfObfuscatedChar',           'ObfuscationRatio',
           'NoOf

## Minimal Processed Dataset

- Dropped four features such as FILENAME, URL, Domain, and Title as LCS code requires numeric, can't consume a text.

In [33]:
# Dropping features that is not suitable for LCS baseline
dropped_features = ["FILENAME", "URL", "Domain", "TLD", "Title", "label"]
classLabel = data["label"]

# All features that the model can use for training in making predictions
X = data.drop(columns=dropped_features) 
# The actual target that the model trying to predict
y = classLabel

X.head(10)

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,...,1,0,0,1,34,20,28,119,0,124
1,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,...,0,0,0,1,50,9,8,39,0,217
2,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,...,0,0,0,1,10,2,7,42,2,5
3,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,...,0,1,1,1,3,27,15,22,1,31
4,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,...,1,1,0,1,244,15,34,72,1,85
5,30,23,0,100.0,1.000000,0.079963,0.060614,3,1,0,...,0,0,0,1,35,1,11,86,0,14
6,25,18,0,100.0,1.000000,0.522907,0.063549,3,1,0,...,0,0,0,1,32,4,14,44,2,17
7,25,18,0,100.0,1.000000,0.522907,0.060486,3,1,0,...,0,0,0,1,24,2,22,36,0,15
8,29,22,0,100.0,1.000000,0.005084,0.056980,2,1,0,...,0,0,0,1,71,4,9,40,1,317
9,18,11,0,100.0,1.000000,0.079963,0.070497,3,1,0,...,0,0,0,1,10,1,12,173,6,65


## Stratified subsample

- Stratified subsample for the data split which aims to preserves the same class distribution as the full dataset.

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape}, Phishing cases: {(y_train == 0).sum()}")
print(f"Test set: {X_test.shape}, Phishing cases: {(y_test == 0).sum()}")
print(f"Training set: {X_train.shape}, Legitimate cases: {(y_train == 1).sum()}")
print(f"Test set: {X_test.shape}, Legitimate cases: {(y_test == 1).sum()}")

Training set: (188636, 50), Phishing cases: 80756
Test set: (47159, 50), Phishing cases: 20189
Training set: (188636, 50), Legitimate cases: 107880
Test set: (47159, 50), Legitimate cases: 26970


- Using the raw data with minimal preprocessing and using a 80/20 data split
- The proportion of legitimate and phishing URLs reflects on what is result of class distribution from Phase-I

## Run the original LCS without any modification

In [37]:
# Initialize eLCS model
model = eLCS(learning_iterations=10000, random_state=42)

# Initiate a start time on how long does the training last.
start = time.time()
model.fit(X_train.to_numpy(), y_train.to_numpy())
elapsed = time.time() - start

prediction = model.predict(X_test.to_numpy())

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy Score: {accuracy_score(y_test, prediction):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, prediction):.4f}")
print(f"Precision Score: {precision_score(y_test, prediction, pos_label=1, zero_division=0):.4f}")
print(f"Recall {recall_score(y_test, prediction, pos_label=1, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, prediction, pos_label=1, zero_division=0):.4f}")
print(f"Precision Score: {precision_score(y_test, prediction, pos_label=0, zero_division=0):.4f}")
print(f"Recall {recall_score(y_test, prediction, pos_label=0, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, prediction, pos_label=0, zero_division=0):.4f}")
print(confusion_matrix(y_test, prediction, labels=[0, 1]))


Training time: 17.07 seconds
Accuracy Score: 0.9990
Balanced Accuracy: 0.9989
Precision Score: 0.9983
Recall 1.0000
F1 Score: 0.9992
Precision Score: 1.0000
Recall 0.9978
F1 Score: 0.9989
[[20144    45]
 [    0 26970]]


# Task 3 Data Preprocessing and Feature Engineering

- From Phase-I and derived from the actual description of the original authors there is no missing values across the dataset.
- But there are inconsistencies within the dataset which was also addressed in Phase-I.
- But because i'm using the original raw dataset from Phase-I, I would start again to avoid any data-leakage

In [75]:
# Start from the raw dataset without dropping any features yet, to addressed any data issues first before any test split
data_lcs = pd.read_csv("/Users/christian/Uni_Files/2026/Semester 2/Data Engineering and AI/Data-Engineering-and-AI-Project-Phase-1/Phase-I/PhiUSIIL_Phishing_URL_Dataset.csv")
# Create a copy of the dataset to preserve the original one.
data_lcs_copy = data_lcs.copy()

# Addressing any missing values
print("\n--- 1. Missing Values ---")
missing = data_lcs_copy.isnull().sum().sum()
print(f"Total missing cells: {missing}")

if missing > 0:
    raise ValueError(
        "Missing values detected."
        "Add an explicit imputation strategy before proceeding."
    )
print("No action needed - dataset has no missing values.\n")

# Addressing URL duplicated records
print("\n--- 2. Duplicated records ---")
before = len(data_lcs_copy)
dup_url_count = data_lcs_copy["URL"].duplicated().sum()
print(f"Duplicate URL values before dedup: {dup_url_count}")

# Drop duplicated records using copy
drop_url_dup = data_lcs_copy.drop_duplicates(subset="URL", keep="first").reset_index(drop=True)
print(f"Rows removed : {before - len(drop_url_dup)}")
print(f"Rows remaining : {len(drop_url_dup):,}\n")


# Addressing invalid values
print("\n--- 3. Invalid values [IsHTTPS and IsDomainIP] ---")

# IsHTTPS: recompute from the actual URL schemed prefix, not the stored flag
before_https_mismatch = (
    (drop_url_dup["IsHTTPS"] == 1) & (~drop_url_dup["URL"].str.startswith("https://"))
).sum()
drop_url_dup["IsHTTPS"] = drop_url_dup["URL"].str.startswith("https://").astype(int)
print(f"IsHTTPS corrected on {before_https_mismatch} rows "
      f"(recomputed from URL scheme prefix). ")

# IsDomainIP: recompute from a strict IPv4 pattern match on Domain
ip_pattern = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")
is_bare_ip = drop_url_dup["Domain"].astype(str).str.match(ip_pattern)
before_ip_mismatch = ((drop_url_dup["IsDomainIP"] == 1) & (~is_bare_ip)).sum()
drop_url_dup["IsDomainIP"] = is_bare_ip.astype(int)
print(f"IsDomainIP corrected on {before_ip_mismatch} rows "
      f"(recomputed from strict IPv4 pattern match on Domain).\n")

# Addressing inconsistent categories and formatting issues
print("\n--- 4. Inconsistent categories & formatting ---")

# Title: the placeholder string "0" only appears when HasTitle=0 and does
# not represents a real page title, so it is cleared to an empty string
placeholder_count = (
    (drop_url_dup["HasTitle"] == 0) & (drop_url_dup["Title"] == "0")
).sum()
drop_url_dup.loc[
    (drop_url_dup["HasTitle"] == 0) & (drop_url_dup["Title"] == "0"), "Title"
] = np.nan
print(f"Title placeholder '0' cleared to NaN on {placeholder_count} rows.")

# Title: strip stray whitespace; TLD: standardise casing to lowercase
drop_url_dup["Title"] = drop_url_dup["Title"].astype(str).str.strip()
drop_url_dup["TLD"] = drop_url_dup["TLD"].astype(str).str.lower()
print("Title whitespace stripped; TLD standardised to lowercase.\n")

# Fixing incorrect / inefficient data types
print("\n--- 5. Data type fixes ---")
binary_cols = [
    "IsDomainIP", "HasObfuscation", "IsHTTPS", "HasTitle", "HasFavicon",
    "Robots", "IsResponsive", "HasDescription", "HasExternalFormSubmit",
    "HasSocialNet", "HasSubmitButton", "HasHiddenFields",
    "HasPasswordField", "Bank", "Pay", "Crypto", "HasCopyrightInfo",
    "label",
]
for col in binary_cols:
    drop_url_dup[col] = drop_url_dup[col].astype("int8")
drop_url_dup["TLD"] = drop_url_dup["TLD"].astype("category")
print(f"Cast {len(binary_cols)} binary columns to int8; TLD to category dtype.\n")

print("\n--- 6. log1p transformations ---")

# Heavily right-skewed count/size columns identified in Phase 1
skewed_cols = [
    "URLLength", "LineOfCode", "LargestLineLength", "NoOfImage", "NoOfCSS",
    "NoOfJS", "NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef",
]

# Safety check: log1p requires values greater than -1
assert (drop_url_dup[skewed_cols] >= 0).all().all(), "Negative values found, log1p not safe"

# Create a _log version of each column, keeping the originals for now
for col in skewed_cols:
    drop_url_dup[f"{col}_log"] = np.log1p(drop_url_dup[col])

print(f"Added log1p columns for {len(skewed_cols)} features")

# Verify: compare the spread before and after
comparison = pd.DataFrame({
    "original_max": drop_url_dup[skewed_cols].max().values,
    "log_max": drop_url_dup[[f"{c}_log" for c in skewed_cols]].max().values,
    "original_skew": drop_url_dup[skewed_cols].skew().values,
    "log_skew": drop_url_dup[[f"{c}_log" for c in skewed_cols]].skew().values,
}, index=skewed_cols).round(3)

print(comparison)



--- 1. Missing Values ---
Total missing cells: 0
No action needed - dataset has no missing values.


--- 2. Duplicated records ---
Duplicate URL values before dedup: 425
Rows removed : 425
Rows remaining : 235,370


--- 3. Invalid values [IsHTTPS and IsDomainIP] ---
IsHTTPS corrected on 493 rows (recomputed from URL scheme prefix). 
IsDomainIP corrected on 39 rows (recomputed from strict IPv4 pattern match on Domain).


--- 4. Inconsistent categories & formatting ---
Title placeholder '0' cleared to NaN on 32612 rows.
Title whitespace stripped; TLD standardised to lowercase.


--- 5. Data type fixes ---
Cast 18 binary columns to int8; TLD to category dtype.


--- 6. log1p transformations ---
Added log1p columns for 9 features
                   original_max  log_max  original_skew  log_skew
URLLength                  6097    8.716         53.347     2.299
LineOfCode               442666   13.001         53.037    -0.476
LargestLineLength      13975732   16.453         47.851     0.106

- Create a copy of the raw dataset, so that the original one is preserved 
- Deriving from the Phase-I there are no missing values across the dataset. But still verified it by checking nulls in every single rows
- Drop 425 duplicated records from URL
- Cross check IsHTTPS with the actual URL scheme prefix and corrected 493 rows
- recomputed IsDomainIP on a strict IPv4 pattern match on Domain and corrected 39 rows
- Addressing inconsistent categories and formatting
- Cast 18 binary columns to int 8 and TLD to category dtype
- Using log1p I transform skewed columns that has been identified from Phase-I, and added a new 9 features derived from the skewed_cols.

## Stratified Train/Test Split

- All data issues that has been identified from Phase-I has been readdressed in Phase-II before any data split, this make sure that no information from the test set influences preprocessing decisions. 
- Also transforming log1p was necessary as this prevents data leakage as it only transforms its own values. So there are no other way to leak any data or any information.

In [85]:
# Separate the target from everything else
target = drop_url_dup["label"]
features = drop_url_dup.drop(columns=["label"])

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=42
)

print(f"Training set: {X_train.shape}, Phishing Case: {(y_train == 0).sum()}")
print(f"Test set: {X_test.shape}, Phishing Case: {(y_test == 0).sum()}")


Training set: (188296, 64), Phishing Case: 80416
Test set: (47074, 64), Phishing Case: 20104


## Dropping identifier / text columns from X_train and X_test

- After cleansing and address every data issues it is safe to drop this columns FILENAME, URL, Domain, and Title.
- This dropping identifier is not considered as predictor it is just needed to be used for proper addressing data issues and transformation.

In [86]:
# Dropping identifier features
identifier_features = ["FILENAME", "URL", "Domain", "TLD", "Title"]

# Keeping URLs aside so any row can still be traced back later
train_urls = X_train["URL"]
test_urls = X_test["URL"]

X_train = X_train.drop(columns=(identifier_features))
X_test = X_test.drop(columns=(identifier_features))

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# 1. Train and test must have exactly the same columns, in the same order
assert list(X_train.columns) == list(X_test.columns), "Train/test columns differ"

# 2. No text columns should remain, since eLCS needs fully numeric input
non_numeric = X_train.select_dtypes(exclude="number").columns.tolist()
print("Non-numeric columns remaining:", non_numeric)   # should print []

# 3. None of the identifiers should still be present
print("Identifiers remaining:", [c for c in identifier_features if c in X_train.columns])   # should print []

Training set: (188296, 59)
Test set: (47074, 59)
Non-numeric columns remaining: []
Identifiers remaining: []


## Outlier Investigation in Training Set Only

 - Outliers can be either errors or genuine information. This reveals what kind of extremes that the dataset have, either a very large website with thousand lines of code or just a a typo, or a broken scrape, or an impossible value like a negative page count.
 

In [88]:
# Outlier Investigation

overall_legit = (y_train == 1).mean()
results = []

for col in skewed_cols:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    is_outlier = X_train[col] > upper

    table = pd.crosstab(is_outlier, y_train)
    chi2, p, _, _ = chi2_contingency(table)

    results.append({
        "feature": col,
        "upper_fence": round(upper, 2),
        "outlier_%": round(is_outlier.mean() * 100, 2),
        "legit_share_in_outliers": round((y_train[is_outlier] == 1).mean(), 4),
        "overall_legit_share": round(overall_legit, 4),
        "chi2_p_value": p,
    })

outlier_summary = pd.DataFrame(results)
print(outlier_summary)

             feature  upper_fence  outlier_%  legit_share_in_outliers  \
0          URLLength         50.5       9.55                   0.0005   
1         LineOfCode       3170.5       8.16                   0.9975   
2  LargestLineLength      19817.5       7.40                   0.6071   
3          NoOfImage         72.5       7.80                   0.9998   
4            NoOfCSS         20.0       7.61                   0.9979   
5             NoOfJS         37.5       5.01                   0.9860   
6        NoOfSelfRef        220.0       6.63                   1.0000   
7       NoOfEmptyRef          2.5      16.52                   0.9698   
8    NoOfExternalRef        141.0       9.66                   0.9998   

   overall_legit_share  chi2_p_value  
0               0.5729  0.000000e+00  
1               0.5729  0.000000e+00  
2               0.5729  2.707740e-17  
3               0.5729  0.000000e+00  
4               0.5729  0.000000e+00  
5               0.5729  0.000000e+0

- Based on this investigation mostly the outliers are entirely legitimate an extreme high values from these features comes from legitimate sites. This matches what we find out in Phase-I.
- 